In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import dlt
import shutil
import os

spark = SparkSession.builder \
    .appName("Read Parquet from ADLS Gen2") \
    .getOrCreate()

In [0]:
storage_account_name = "wajbah"
storage_account_key = "iPxysh7QvmA2sU5XcPmn9oSt3ssPYGUqPQdC5eDwN5T6/TE9SnLOit7/YL8p+uMXDfDeTAnihW81+AStWiR4+w=="
container_name = "wajbah"

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
Chefs_source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze/Chefs"
ChefPromoCode_source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze/ChefPromocode"
Companies_source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze/Companies"
Customers_source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze/Customers"
Itemrate_source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze/Itemrate"
menuitems_source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze/menuitems"
Order_source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze/Order"
OrderMenuItems_source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze/OrderMenuitems"
Promocode_source_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze/Promocode"

In [0]:
# Static DataFrames to infer schemas
static_chefs_df = spark.read.format("parquet").load(Chefs_source_path)
static_chefpromocode_df = spark.read.format("parquet").load(ChefPromoCode_source_path)
static_companies_df = spark.read.format("parquet").load(Companies_source_path)
static_customers_df = spark.read.format("parquet").load(Customers_source_path)
static_itemrate_df = spark.read.format("parquet").load(Itemrate_source_path)
static_menuitems_df = spark.read.format("parquet").load(menuitems_source_path)
static_order_df = spark.read.format("parquet").load(Order_source_path)
static_ordermenuitems_df = spark.read.format("parquet").load(OrderMenuItems_source_path)
static_promocode_df = spark.read.format("parquet").load(Promocode_source_path)

# Infer schemas from static DataFrames
chefs_schema = static_chefs_df.schema
chefpromocode_schema = static_chefpromocode_df.schema
companies_schema = static_companies_df.schema
customers_schema = static_customers_df.schema
itemrate_schema = static_itemrate_df.schema
menuitems_schema = static_menuitems_df.schema
order_schema = static_order_df.schema
ordermenuitems_schema = static_ordermenuitems_df.schema
promocode_schema = static_promocode_df.schema

In [0]:
# Define the Bronze Table as a streaming live table
@dlt.table(
  name="chefs_bronze",
  comment="Live Bronze Table For Chefs",
  table_properties={
    "quality": "bronze"
  }
)
def chefs_bronze_table():
    # Read from the streaming source table
    bronze_df = spark.readStream.schema(chefs_schema).format("parquet").load(Chefs_source_path)
    bronze_df.createOrReplaceTempView("Chefs_bronze")
    return bronze_df

Name,Type
ChefId,string
PhoneNumber,int
Email,string
Password,string
ChefFirstName,string
ChefLastName,string
RestaurantName,string
BirthDate,timestamp
Description,string
Rating,double


In [0]:
# Define the Bronze Table as a streaming live table
@dlt.table(
  name="chefpromo_bronze",
  comment="Live Bronze Table For Chef Promo Code",
  table_properties={
    "quality": "bronze"
  }
)
def chefpromo_bronze():
    # Read from the streaming source table
    bronze_df = spark.readStream.schema(chefpromocode_schema).format("parquet").load(ChefPromoCode_source_path)
    bronze_df.createOrReplaceTempView("chefpromo_bronze")
    return bronze_df

Name,Type
PromoCodeId,int
ChefId,string


In [0]:
# Define the Bronze Table as a streaming live table
@dlt.table(
  name="promocode_bronze",
  comment="Live Bronze Table For Promo Code",
  table_properties={
    "quality": "bronze"
  }
)
def promocode_bronze():
    # Read from the streaming source table
    bronze_df = spark.readStream.schema(promocode_schema).format("parquet").load(Promocode_source_path)
    bronze_df.createOrReplaceTempView("promocode_bronze")
    return bronze_df

Name,Type
PromoCodeId,int
Name,string
StartDate,timestamp
ExpireDate,timestamp
DiscountPercentage,"decimal(18,2)"
MaxUsers,int
MaxLimit,int
MinLimit,int


In [0]:
# Define the Silver Table as a streaming live table
@dlt.table(
  name="chefs_silver",
  comment="Live Silver Table For Chefs",
  table_properties={
    "quality": "silver"
  }
)
def chefs_silver():
    # Read from the Bronze tables as streaming sources
    chefs_bronze_df = dlt.read_stream("chefs_bronze")
   
    # Join chefs_bronze_df with ChefPromoCode_BronzeTable and PromoCode_BronzeTable using SQL
    silver_df = spark.sql("""
        SELECT 
            cb.ChefId,
            CONCAT_WS(' ', cb.ChefFirstName, cb.ChefLastName) AS Chef_name,
            cb.RestaurantName,
            DAY(cb.BirthDate) AS BirthDay,
            MONTH(cb.BirthDate) AS BirthMonth,
            YEAR(cb.BirthDate) AS BirthYear,
            cb.Rating,
            cb.Governorate,
            cb.City,
            cb.Role,
            cb.Active,
            datediff(current_date(), cb.BirthDate) AS Age 
        FROM 
            chefs_bronze cb
    """)
    silver_df.createOrReplaceTempView("chefs_silver")
    return silver_df

Name,Type
ChefId,string
Chef_name,string
RestaurantName,string
BirthDay,int
BirthMonth,int
BirthYear,int
Rating,double
Governorate,string
City,string
Role,string


In [0]:
# Define the Silver Table as a streaming live table
@dlt.table(
  name="promocode_silver",
  comment="Live Silver Table For PromoCode",
  table_properties={
    "quality": "silver"
  }
)
def promocode_silver():
    # Read from the Bronze tables as streaming sources
    promocode_bronze_df = dlt.read_stream("promocode_bronze")
    
    # Join chefs_bronze_df with ChefPromoCode_BronzeTable and PromoCode_BronzeTable using SQL
    silver_df = spark.sql("""
        SELECT 
        *
        FROM 
            promocode_bronze 
    """)
    silver_df.createOrReplaceTempView("promocode_silver")
    return silver_df


Name,Type
PromoCodeId,int
Name,string
StartDate,timestamp
ExpireDate,timestamp
DiscountPercentage,"decimal(18,2)"
MaxUsers,int
MaxLimit,int
MinLimit,int


In [0]:
# Define the Silver Table as a streaming live table
@dlt.table(
  name="chefpromo_silver",
  comment="Live Silver Table For PromoCode",
  table_properties={
    "quality": "silver"
  }
)
def chefpromo_silver():
    # Read from the Bronze tables as streaming sources
    promocode_bronze_df = dlt.read_stream("chefpromo_bronze")
    
    # Join chefs_bronze_df with ChefPromoCode_BronzeTable and PromoCode_BronzeTable using SQL
    silver_df = spark.sql("""
        SELECT 
        *
        FROM 
            chefpromo_bronze 
    """)
    silver_df.createOrReplaceTempView("chefpromo_silver")
    return silver_df

Name,Type
PromoCodeId,int
ChefId,string


In [0]:
@dlt.table(
  name="Companies_bronze",
  comment="Live Bronze Table For Chef Companies",
  table_properties={
    "quality": "bronze"
  }
)
def Companies_bronze():
    # Read from the streaming source table
    bronze_df = spark.readStream.schema(companies_schema).format("parquet").load(Companies_source_path)
    bronze_df.createOrReplaceTempView("Companies_bronze")
    return bronze_df

Name,Type
CompanyId,int
CompanyName,string
Wallet,"decimal(18,2)"
Email,string
Password,string
PhoneNumber,int
DeliveryFees,"decimal(18,2)"
contract,string
Area,string


In [0]:
# Define the Silver Table as a streaming live table
@dlt.table(  
  name="Companies_silver",
  comment="Live Silver Table For Companies",
  table_properties={
    "quality": "silver"
  }
)
def Companies_silver():
    # Read from the Bronze tables as streaming sources
    Companies_bronze_df = dlt.read_stream("Companies_bronze")
    
    # Join chefs_bronze_df with ChefPromoCode_BronzeTable and PromoCode_BronzeTable using SQL
    silver_df = spark.sql("""
        SELECT 
            CompanyId,
            CompanyName,
            Wallet,
            DeliveryFees,
            Area
        FROM 
            Companies_bronze 
    """)
    silver_df.createOrReplaceTempView("Companies_silver")
    return silver_df

Name,Type
CompanyId,int
CompanyName,string
Wallet,"decimal(18,2)"
DeliveryFees,"decimal(18,2)"
Area,string


In [0]:
# Define the Bronze Table as a streaming live table
@dlt.table(
  name="Customers_bronze",
  comment="Live Bronze Table For Customers",
  table_properties={
    "quality": "bronze"
  }
)
def Customers_bronze():
    # Read from the streaming source table
    bronze_df = spark.readStream.schema(customers_schema).format("parquet").load(Customers_source_path)
    bronze_df.createOrReplaceTempView("Customers_bronze")
    return bronze_df

Name,Type
CustomerId,int
PhoneNumber,int
Email,string
Password,string
FirstName,string
LastName,string
BirthDate,timestamp
Wallet,"decimal(18,2)"
UsedCoupones,string
Role,string


In [0]:
# Define the Silver Table as a streaming live table
@dlt.table(
  name="Customers_silver",
  comment="Live Silver Table For Customers",
  table_properties={
    "quality": "silver"
  }
)
def Customers_silver():
    # Read from the Bronze tables as streaming sources
    Customers_bronze_df = dlt.read_stream("Customers_bronze")
    
    silver_df = spark.sql("""
        SELECT 
            CustomerId,
            concat(FirstName,' ', LastName) AS Name,
            Wallet,
            DAY(BirthDate) AS BirthDay,
            MONTH(BirthDate) AS BirthMonth,
            YEAR(BirthDate) AS BirthYear,
            datediff(current_date(), BirthDate) AS Age ,
            UsedCoupones,
            Role,
            Favourites,
            State
        FROM 
            Customers_bronze 
    """)
    silver_df.createOrReplaceTempView("Customers_silver")
    return silver_df

Name,Type
CustomerId,int
Name,string
Wallet,"decimal(18,2)"
BirthDay,int
BirthMonth,int
BirthYear,int
Age,int
UsedCoupones,string
Role,string
Favourites,string


In [0]:
# Define the Bronze Table as a streaming live table
@dlt.table(
  name="menuitems_bronze",
  comment="Live Bronze Table For menuitems",
  table_properties={
    "quality": "bronze"
  }
)
def menuitems_bronze():
    # Read from the streaming source table
    bronze_df = spark.readStream.schema(menuitems_schema).format("parquet").load(menuitems_source_path)
    bronze_df.createOrReplaceTempView("menuitems_bronze")
    return bronze_df

Name,Type
MenuItemId,int
Name,string
Category,string
Occassions,string
EstimatedTime,string
OrderingTime,string
HealthyMode,boolean
Description,string
Photo,string
CreatedOn,timestamp


In [0]:
# Define the Silver Table as a streaming live table
@dlt.table(
  name="menuitems_silver",
  comment="Live Silver Table For menuitems",
  table_properties={
    "quality": "silver"
  }
)
def menuitems_silver():
    # Read from the Bronze tables as streaming sources
    menuitems_bronze_df = dlt.read_stream("menuitems_bronze")
    
    silver_df = spark.sql("""
        SELECT 
            MenuItemId,
            Category,
            Occassions,
            EstimatedTime,
            OrderingTime,
            HealthyMode,
            CreatedOn ,
            UpdatedOn,
            ChefId,
            PriceLarge,
            PriceMedium,
            PriceSmall,
            Rate
        FROM 
            menuitems_bronze 
    """)
    silver_df.createOrReplaceTempView("menuitems_silver")
    return silver_df

Name,Type
MenuItemId,int
Category,string
Occassions,string
EstimatedTime,string
OrderingTime,string
HealthyMode,boolean
CreatedOn,timestamp
UpdatedOn,timestamp
ChefId,string
PriceLarge,"decimal(18,2)"


In [0]:
# Define the Bronze Table as a streaming live table
@dlt.table(
  name="Order_bronze",
  comment="Live Bronze Table For Order",
  table_properties={
    "quality": "bronze"
  }
)
def Order_bronze():
    # Read from the streaming source table
    bronze_df = spark.readStream.schema(order_schema).format("parquet").load(Order_source_path)
    bronze_df.createOrReplaceTempView("Order_bronze")
    return bronze_df

Name,Type
OrderId,int
Notes,string
DeliveryFees,"decimal(18,2)"
DeliveryNumber,int
CreatedOn,timestamp
DeliveryTime,timestamp
Status,string
Copoun,string
CashDelivered,boolean
EstimatedTime,string


In [0]:
# Define the Silver Table as a streaming live table
@dlt.table(
  name="Order_silver",
  comment="Live Silver Table For Order",
  table_properties={
    "quality": "silver"
  }
)
def Order_silver():
    # Read from the Bronze tables as streaming sources
    Order_bronze_df = dlt.read_stream("Order_bronze")
    
    silver_df = spark.sql("""
        SELECT 
            OrderId,
            DeliveryFees,
            CreatedOn,
            DeliveryTime,
            Status,
            Copoun ,
            CashDelivered,
            EstimatedTime,
            CompanyId,
            CustomerId,
            SubTotal,
            TotalPrice,
            ChefId
        FROM 
            Order_bronze 
    """)
    silver_df.createOrReplaceTempView("Order_silver")
    return silver_df

Name,Type
OrderId,int
DeliveryFees,"decimal(18,2)"
CreatedOn,timestamp
DeliveryTime,timestamp
Status,string
Copoun,string
CashDelivered,boolean
EstimatedTime,string
CompanyId,int
CustomerId,int


In [0]:
@dlt.table(
  name="OrderMenuItems_bronze",
  comment="Live Bronze Table For OrderMenuItems",
  table_properties={
    "quality": "bronze"
  }
)
def OrderMenuItems_bronze():
    # Read from the streaming source table
    bronze_df = spark.readStream.schema(ordermenuitems_schema).format("parquet").load(OrderMenuItems_source_path)
    bronze_df.createOrReplaceTempView("OrderMenuItems_bronze")
    return bronze_df

Name,Type
OrderId,int
MenuItemId,int
Quantity,int
Size,string


In [0]:
@dlt.table(
  name= "OrderMenuItems_silver",
  comment="Live Silver Table For OrderMenuItems",
  table_properties={
    "quality": "silver"
  }
)
def OrderMenuItems_silver():
    # Read from the Bronze tables as streaming sources
    Order_bronze_df = dlt.read_stream("OrderMenuItems_bronze")
    menuitems_bronze_df = dlt.read_stream("menuitems_bronze")
    
    silver_df = spark.sql("""
SELECT 
    om.*,
    CASE
        WHEN om.size = 'Large' THEN m.PriceLarge
        WHEN om.size = 'Medium' THEN m.PriceMedium
        WHEN om.size = 'Small' THEN m.PriceSmall
    END AS Price,
    om.quantity * 
    CASE
        WHEN om.size = 'Large' THEN m.PriceLarge
        WHEN om.size = 'Medium' THEN m.PriceMedium
        WHEN om.size = 'Small' THEN m.PriceSmall
    END AS TotalPrice
FROM 
    OrderMenuItems_bronze AS om
JOIN
    menuitems_bronze AS m
ON
    om.MenuItemId = m.MenuItemId
    """)
    silver_df.createOrReplaceTempView("OrderMenuItems_silver")
    return silver_df 


Name,Type
OrderId,int
MenuItemId,int
Quantity,int
Size,string
Price,"decimal(18,2)"
TotalPrice,"decimal(29,2)"


In [0]:
@dlt.table(
  name="Itemrate_bronze",
  comment="Live Bronze Table For OrderMenuItems",
  table_properties={
    "quality": "bronze"
  }
)
def Itemrate_bronze():
    # Read from the streaming source table
    bronze_df = spark.readStream.schema(itemrate_schema).format("parquet").load(Itemrate_source_path)
    bronze_df.createOrReplaceTempView("Itemrate_bronze")
    return bronze_df

Name,Type
CustomerId,int
MenuItemId,int
Rating,double


In [0]:
# Define the Silver Table as a streaming live table
@dlt.table(
  name="Itemrate_silver",
  comment="Live Silver Table For Itemrate",
  table_properties={
    "quality": "silver"
  }
)
def Itemrate_silver():
    # Read from the Bronze tables as streaming sources
    promocode_bronze_df = dlt.read_stream("Itemrate_bronze")
    
    # Join chefs_bronze_df with ChefPromoCode_BronzeTable and PromoCode_BronzeTable using SQL
    silver_df = spark.sql("""
        SELECT 
        *
        FROM 
            Itemrate_bronze 
    """)
    silver_df.createOrReplaceTempView("Itemrate_silver")
    return silver_df

Name,Type
CustomerId,int
MenuItemId,int
Rating,double


In [0]:
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints"
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/Chefs"

# Function to delete directory in ADLS Gen2
def delete_adls_directory(path):
    dbutils.fs.rm(path, True)

# Function to delete parquet files in a given directory
def delete_parquet_files(path):
    files = dbutils.fs.ls(path)
    for file in files:
        if file.path.endswith(".parquet"):
            dbutils.fs.rm(file.path, False)

# Delete the checkpoints directory
delete_adls_directory(checkpoint_location)


In [0]:

# Read from the Delta table created by DLT
silver_df = spark.readStream.format("delta").table("chefs_silver")

# Define the writeStream operation with the checkpoint location specified
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints/chefs_silver_table"
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/Chefs"

# Delete all parquet files in the output directory
delete_parquet_files(output_path)

streaming_query = silver_df.writeStream \
    .outputMode("append") \
    .format("Delta") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_location) \
    .start()


In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/promocode_silver"
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints/promocode_silver"

# Read from the Delta table created by DLT
silver_df = spark.readStream.format("delta").table("promocode_silver")

# Delete all parquet files in the output directory
delete_parquet_files(output_path)

# Define the writeStream operation with the checkpoint location specified
streaming_query = silver_df.writeStream \
    .outputMode("append") \
    .format("Delta") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_location) \
    .start()

In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/chefpromo_silver"
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints/chefpromo_silver"

# Read from the Delta table created by DLT
silver_df = spark.readStream.format("delta").table("chefpromo_silver")

# Delete all parquet files in the output directory
delete_parquet_files(output_path)

# Define the writeStream operation with the checkpoint location specified
streaming_query = silver_df.writeStream \
    .outputMode("append") \
    .format("Delta") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_location) \
    .start()

In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/Companies_silver"
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints/Companies_silver"

# Read from the Delta table created by DLT
silver_df = spark.readStream.format("delta").table("Companies_silver")

# Delete all parquet files in the output directory
delete_parquet_files(output_path)

# Define the writeStream operation with the checkpoint location specified
streaming_query = silver_df.writeStream \
    .outputMode("append") \
    .format("delta") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_location) \
    .start()


In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/Customers_silver"
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints/Customers_silver"

# Read from the Delta table created by DLT
silver_df = spark.readStream.format("delta").table("Customers_silver")

# Delete all parquet files in the output directory
delete_parquet_files(output_path)

# Define the writeStream operation with the checkpoint location specified
streaming_query = silver_df.writeStream \
    .outputMode("append") \
    .format("Delta") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_location) \
    .start()

In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/menuitems_silver"
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints/menuitems_silver"

# Read from the Delta table created by DLT
silver_df = spark.readStream.format("delta").table("menuitems_silver")

# Delete all parquet files in the output directory
delete_parquet_files(output_path)

# Define the writeStream operation with the checkpoint location specified
streaming_query = silver_df.writeStream \
    .outputMode("append") \
    .format("Delta") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_location) \
    .start()

In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/Order_silver"
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints/Order_silver"

# Read from the Delta table created by DLT
silver_df = spark.readStream.format("delta").table("Order_silver")

# Delete all parquet files in the output directory
delete_parquet_files(output_path)

# Define the writeStream operation with the checkpoint location specified
streaming_query = silver_df.writeStream \
    .outputMode("append") \
    .format("Delta") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_location) \
    .option("mergeSchema", "true") \
    .start()

In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/OrderMenuItems_silver"
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints/OrderMenuitems_silver"

# Read from the Delta table created by DLT
silver_df = spark.readStream.format("delta").table("ordermenuitems_silver")

# Delete all parquet files in the output directory
delete_parquet_files(output_path)

# Define the writeStream operation with the checkpoint location specified
streaming_query = silver_df.writeStream \
    .outputMode("append") \
    .format("Delta") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_location) \
    .start()

In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/Itemrate_silver"
checkpoint_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver/checkpoints/Itemrate_silver"

# Read from the Delta table created by DLT
silver_df = spark.readStream.format("delta").table("Itemrate_silver")

# Delete all parquet files in the output directory
delete_parquet_files(output_path)

# Define the writeStream operation with the checkpoint location specified
streaming_query = silver_df.writeStream \
    .outputMode("append") \
    .format("Delta") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_location) \
    .start()